In [1]:
from google.colab import files
import zipfile
import os

uploaded = files.upload()

zip_name = next(
    name for name in uploaded
    if name.lower().endswith(".zip")
)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/trendpulse_data")

for root, dirs, filenames in os.walk("/content/trendpulse_data"):
    for filename in filenames:
        if filename.lower().endswith((".csv", ".json")):
            print(os.path.join(root, filename))

Saving c2MzB3IbbJgdaIre.zip to c2MzB3IbbJgdaIre.zip
/content/trendpulse_data/dataset/USvideos.csv
/content/trendpulse_data/dataset/categories.csv
/content/trendpulse_data/dataset/GBvideos.csv
/content/trendpulse_data/dataset/INvideos.csv
/content/trendpulse_data/__MACOSX/dataset/._USvideos.csv
/content/trendpulse_data/__MACOSX/dataset/._INvideos.csv
/content/trendpulse_data/__MACOSX/dataset/._categories.csv
/content/trendpulse_data/__MACOSX/dataset/._GBvideos.csv


In [2]:
from pathlib import Path
from datetime import datetime
import pandas as pd
from collections import Counter

DATA_DIR = Path("/content/trendpulse_data")

COUNTRIES = {
    "US": "USvideos.csv",
    "GB": "GBvideos.csv",
    "IN": "INvideos.csv"
}

REQUIRED_COLUMNS = [
    "video_id", "trending_date", "title", "channel_title",
    "category_id", "publish_time", "views", "likes",
    "dislikes", "comment_count", "comments_disabled",
    "ratings_disabled", "video_error_or_removed"
]

VIEW_THRESHOLD = 1_000_000
LIKE_RATIO_THRESHOLD = 0.80

print(f"TrendPulse | Started: {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Countries selected: {', '.join(COUNTRIES.keys())}")

TrendPulse | Started: 2026-11-02 06:56:55
Countries selected: US, GB, IN


In [3]:
def find_file(filename):
    matches = list(DATA_DIR.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"File not found: {filename}")
    return matches[0]


def load_country(path, country, max_rows=None):
    """Load a country's CSV and return a list of dictionaries."""
    df = pd.read_csv(path, nrows=max_rows)

    missing = set(REQUIRED_COLUMNS) - set(df.columns)
    if missing:
        raise ValueError(
            f"{country}: Missing required columns: {sorted(missing)}"
        )

    df["country"] = country
    records = df.to_dict("records")

    print(f"{country}: {len(records):,} records loaded")
    return records


all_records = []

for country, filename in COUNTRIES.items():
    path = find_file(filename)
    country_records = load_country(path, country)
    all_records.extend(country_records)

print(f"Total records: {len(all_records):,}")

assert len(all_records) >= 40_000, (
    "Need at least 40,000 records. Check the selected CSV files."
)

US: 16,080 records loaded
GB: 15,075 records loaded
IN: 15,075 records loaded
Total records: 46,230


In [4]:
category_path = find_file("categories.csv")
categories_df = pd.read_csv(category_path)

print("Category columns:", categories_df.columns.tolist())
print(categories_df.head())

Category columns: ['category_id', 'category_name']
   category_id     category_name
0            1  Film & Animation
1            2  Autos & Vehicles
2           10             Music
3           15    Pets & Animals
4           17            Sports


In [5]:
name_column = (
    "category_name"
    if "category_name" in categories_df.columns
    else "title"
)

category_names = dict(
    zip(
        categories_df["category_id"].astype(str),
        categories_df[name_column]
    )
)

print("Number of categories:", len(category_names))
print(category_names)

Number of categories: 17
{'1': 'Film & Animation', '2': 'Autos & Vehicles', '10': 'Music', '15': 'Pets & Animals', '17': 'Sports', '19': 'Travel & Events', '20': 'Gaming', '22': 'People & Blogs', '23': 'Comedy', '24': 'Entertainment', '25': 'News & Politics', '26': 'Howto & Style', '27': 'Education', '28': 'Science & Technology', '29': 'Nonprofits & Activism', '30': 'Movies', '43': 'Shows'}


In [6]:
NUMERIC_FIELDS = ["views", "likes", "dislikes", "comment_count"]

BOOLEAN_FIELDS = [
    "comments_disabled",
    "ratings_disabled",
    "video_error_or_removed"
]

parse_failures = Counter()


def to_int(value, field):
    try:
        if pd.isna(value) or str(value).strip() == "":
            parse_failures[field] += 1
            return None

        return int(float(value))
    except (ValueError, TypeError, OverflowError):
        parse_failures[field] += 1
        return None


def to_bool(value, field):
    if isinstance(value, bool):
        return value

    text = str(value).strip().lower()

    if text in {"true", "1", "yes"}:
        return True
    if text in {"false", "0", "no"}:
        return False

    parse_failures[field] += 1
    return None


def parse_trending_date(value):
    try:
        return datetime.strptime(str(value), "%y.%d.%m")
    except (ValueError, TypeError):
        parse_failures["trending_date"] += 1
        return None


def parse_publish_time(value):
    try:
        return pd.to_datetime(value, errors="raise", utc=True)
    except (ValueError, TypeError):
        parse_failures["publish_time"] += 1
        return None


for record in all_records:
    for field in NUMERIC_FIELDS:
        record[field] = to_int(record.get(field), field)

    for field in BOOLEAN_FIELDS:
        record[field] = to_bool(record.get(field), field)

    record["trending_date"] = parse_trending_date(
        record.get("trending_date")
    )
    record["publish_time"] = parse_publish_time(
        record.get("publish_time")
    )

print("Parse failures by field:")
for field, count in sorted(parse_failures.items()):
    print(f"{field}: {count}")

Parse failures by field:
comment_count: 36
likes: 12


In [8]:
print("First record:", all_records[0])

first = all_records[0]
video_tuple = (
    first["video_id"],
    first["trending_date"],
    first["views"],
    first["likes"]
)
print("Tuple:", video_tuple)

ids_by_country = {}

for country in COUNTRIES:
    ids_by_country[country] = {
        r["video_id"] for r in all_records
        if r["country"] == country and r["video_id"]
    }

for country, ids in ids_by_country.items():
    print(country, "unique video IDs:", len(ids))

us_gb = ids_by_country["US"] & ids_by_country["GB"]
print("Videos in both US and GB:", len(us_gb))

us_only = ids_by_country["US"] - ids_by_country["GB"]
print("Videos only in US compared with GB:", len(us_only))

First record: {'video_id': 'g_N-w8LeAi8', 'trending_date': datetime.datetime(2017, 11, 14, 0, 0), 'title': 'Honest Football Prank LEAKED', 'channel_title': 'QuietSpeech10', 'category_id': 22, 'publish_time': Timestamp('2017-11-14 07:04:28+0000', tz='UTC'), 'tags': 'Monsoon|Robots', 'views': 48555, 'likes': 2189, 'dislikes': 180, 'comment_count': 136, 'thumbnail_link': 'https://i.ytimg.com/vi/g_N-w8LeAi8/default.jpg', 'comments_disabled': False, 'ratings_disabled': False, 'video_error_or_removed': False, 'description': 'Watch this video. Subscribe for more content about chess. Business enquiries in the channel about page.', 'country': 'US'}
Tuple: ('g_N-w8LeAi8', datetime.datetime(2017, 11, 14, 0, 0), 48555, 2189)
US unique video IDs: 5068
GB unique video IDs: 4113
IN unique video IDs: 5878
Videos in both US and GB: 134
Videos only in US compared with GB: 4934


In [9]:
channel_counts = Counter(
    r["channel_title"]
    for r in all_records
    if r.get("channel_title")
)

top_15_channels = channel_counts.most_common(15)

for rank, (channel, appearances) in enumerate(top_15_channels, start=1):
    print(f"{rank}. {channel}: {appearances} appearances")

1. QuietChallenge75: 2464 appearances
2. OfficialReview41: 1586 appearances
3. UltimateUnboxing65: 1433 appearances
4. InsaneReaction46: 1233 appearances
5. Brand NewChallenge45: 1175 appearances
6. CrazyTrailer91: 1018 appearances
7. FinalTrailer17: 911 appearances
8. UltimateSpeech20: 770 appearances
9. Brand NewSecret51: 684 appearances
10. WildHighlights7: 682 appearances
11. SurprisingHighlights41: 650 appearances
12. FinalBreakdown11: 643 appearances
13. CrazyRecipe67: 603 appearances
14. QuietInterview9: 570 appearances
15. UltimateVlog96: 565 appearances


In [10]:
def normalize_category_id(value):
    try:
        return str(int(float(value)))
    except (ValueError, TypeError):
        return str(value).strip()


category_summary = {}

for record in all_records:
    country = record["country"]
    category_id = normalize_category_id(record["category_id"])
    category_name = category_names.get(category_id, "Unknown")

    if country not in category_summary:
        category_summary[country] = {}

    if category_name not in category_summary[country]:
        category_summary[country][category_name] = 0

    category_summary[country][category_name] += 1


for country, categories in category_summary.items():
    print(f"\nCountry: {country}")
    for category, count in list(categories.items())[:5]:
        print(f"  {category}: {count}")


Country: US
  People & Blogs: 1423
  News & Politics: 755
  Music: 3052
  Education: 561
  Entertainment: 4371

Country: GB
  Science & Technology: 470
  Howto & Style: 588
  Comedy: 941
  Entertainment: 2844
  Music: 5013

Country: IN
  Music: 2298
  Film & Animation: 770
  Entertainment: 5117
  Comedy: 966
  People & Blogs: 1098


In [11]:
project_name = "TrendPulse"  # Global variable


def scope_demo():
    local_message = "Inside the function"
    print(project_name)
    print(local_message)


scope_demo()

print(project_name)

try:
    print(local_message)
except NameError:
    print("local_message is not available outside the function")

TrendPulse
Inside the function
TrendPulse
local_message is not available outside the function


In [12]:
def title_features(title):
    title = str(title or "")
    words = title.split()

    uppercase_words = sum(
        1 for word in words
        if word.isupper() and any(ch.isalpha() for ch in word)
    )

    punctuation_count = 0

    # range() with index access
    for i in range(len(title)):
        if title[i] in "!?":
            punctuation_count += 1

    return {
        "uppercase_words": uppercase_words,
        "question_exclamation_marks": punctuation_count,
        "word_count": len(words)
    }


print(title_features(all_records[0]["title"]))

{'uppercase_words': 1, 'question_exclamation_marks': 0, 'word_count': 4}


In [13]:
index = 0
first_high_view_video = None

while index < len(all_records):
    record = all_records[index]
    index += 1

    if record["views"] is None:
        continue  # Skip invalid views

    if record["views"] >= VIEW_THRESHOLD:
        first_high_view_video = record
        break  # Stop after the first matching record

if first_high_view_video:
    print("First video above threshold:")
    print(first_high_view_video["title"])
    print(first_high_view_video["views"])
else:
    print("No video met the view threshold.")

First video above threshold:
Official Cats Speech EXCLUSIVE
2841278


In [14]:
def classify_video(record):
    views = record.get("views")
    likes = record.get("likes")
    dislikes = record.get("dislikes")

    # Missing data or unrated records
    if record.get("ratings_disabled") is True:
        return "Unrated"

    if views is None or likes is None or dislikes is None:
        return "Unrated"

    if views < 0 or likes < 0 or dislikes < 0:
        return "Unrated"

    total_ratings = likes + dislikes

    if total_ratings == 0:
        like_ratio = None
    else:
        like_ratio = likes / total_ratings

    if views == 0:
        return "Low Engagement"

    if (
        views >= VIEW_THRESHOLD
        and like_ratio is not None
        and like_ratio >= LIKE_RATIO_THRESHOLD
    ):
        return "Viral Hit"

    elif (
        views >= 100_000
        and like_ratio is not None
        and like_ratio < 0.50
    ):
        return "Divisive"

    elif views >= 100_000:
        return "Steady Performer"

    else:
        return "Low Engagement"


label_counts = Counter(
    classify_video(record) for record in all_records
)

for label, count in sorted(label_counts.items()):
    print(f"{label}: {count:,}")

Divisive: 27
Low Engagement: 8,734
Steady Performer: 27,416
Unrated: 556
Viral Hit: 9,497


In [15]:
example = {
    "views": 1_500_000,
    "likes": 90_000,
    "dislikes": 10_000,
    "ratings_disabled": False
}

print(classify_video(example))

Viral Hit
